# 📖 Lab 1: Basic Web Crawler

Our functional requirements: **Crawl the web from seed URLs** and **extract text data** for LLM training.

## 🏗️ Architecture — Starting Point

```
Seed URLs ──> ┌──────────────┐     ┌───────────┐     ┌──────────────┐
              │ Frontier     │────>│  Crawler   │────>│  S3 / Local  │
              │ Queue        │<────│  Worker    │     │  Text Data   │
              └──────────────┘     └───────┬───┘     └──────────────┘
               new URLs added              │
               back to queue         DNS → fetch HTML
                                     → extract text
                                     → extract URLs
```

This is the simplest possible crawler — one queue, one worker, no fault tolerance, no politeness. We'll improve in later labs.

## Learning Objectives

- Build a working crawler that follows links
- Understand the frontier queue (BFS vs DFS)
- Extract text and discover new URLs from real HTML
- See why this naive approach needs improvements (scale, failures, politeness)

## 🛠️ Setup

```bash
cd system-designs/web-crawler
docker-compose up -d
```

Select the **"Web Crawler (Python)"** kernel.

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from collections import deque
import time

print("✅ Dependencies loaded.")

## 🔧 The Data Flow: Step by Step

Let's build the crawler step by step, following the data flow:
1. Fetch HTML from a URL
2. Extract text content
3. Extract linked URLs
4. Add new URLs to the frontier queue
5. Repeat

In [ ]:
def fetch_html(url: str, timeout: int = 10) -> str:
    """Step 1: Fetch the raw HTML from a URL."""
    try:
        resp = requests.get(url, timeout=timeout, headers={
            "User-Agent": "EducationalCrawlerBot/1.0 (system-design-labs)"
        })
        resp.raise_for_status()
        return resp.text
    except requests.RequestException as e:
        print(f"  ❌ Failed to fetch {url}: {e}")
        return ""


def extract_text(html: str) -> str:
    """Step 2: Extract visible text content from HTML."""
    soup = BeautifulSoup(html, "html.parser")
    # Remove script and style elements
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()
    text = soup.get_text(separator="\n", strip=True)
    # Clean up: remove excessive blank lines
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines)


def extract_urls(html: str, base_url: str) -> list[str]:
    """Step 3: Extract all linked URLs from the HTML."""
    soup = BeautifulSoup(html, "html.parser")
    urls = []
    for link in soup.find_all("a", href=True):
        href = link["href"]
        absolute_url = urljoin(base_url, href)
        parsed = urlparse(absolute_url)
        # Only keep http/https URLs, skip fragments and non-web links
        if parsed.scheme in ("http", "https"):
            clean_url = f"{parsed.scheme}://{parsed.netloc}{parsed.path}"
            urls.append(clean_url)
    return list(set(urls))  # deduplicate


# Test each step on a real page
TEST_URL = "https://example.com"

print(f"🔧 Step-by-step on {TEST_URL}:\n")

# Step 1: Fetch
html = fetch_html(TEST_URL)
print(f"  📥 Fetched {len(html)} bytes of HTML")

# Step 2: Extract text
text = extract_text(html)
print(f"  📝 Extracted {len(text)} chars of text:")
print(f"     '{text[:100]}...'")

# Step 3: Extract URLs
urls = extract_urls(html, TEST_URL)
print(f"  🔗 Found {len(urls)} URLs:")
for u in urls[:5]:
    print(f"     {u}")

## 🕸️ The Complete Crawler: BFS with Frontier Queue

Now let's put it all together. The frontier queue is a FIFO queue (BFS — breadth-first search) — we crawl all pages at depth N before moving to depth N+1. This ensures we cover a broad set of domains before going deep into any single site.

In [ ]:
def crawl(seed_urls: list[str], max_pages: int = 10) -> dict:
    """
    Basic BFS web crawler.
    
    Frontier queue = deque (FIFO).
    Visited set = prevents re-crawling.
    """
    frontier = deque(seed_urls)
    visited: set[str] = set()
    results = []

    while frontier and len(results) < max_pages:
        url = frontier.popleft()

        if url in visited:
            continue
        visited.add(url)

        print(f"  🌐 [{len(results)+1}/{max_pages}] Crawling: {url}")

        # Fetch
        html = fetch_html(url)
        if not html:
            continue

        # Extract text
        text = extract_text(html)

        # Extract and enqueue new URLs
        new_urls = extract_urls(html, url)
        new_count = 0
        for new_url in new_urls:
            if new_url not in visited:
                frontier.append(new_url)
                new_count += 1

        results.append({
            "url": url,
            "text_length": len(text),
            "text_preview": text[:80],
            "urls_discovered": len(new_urls),
            "new_urls_enqueued": new_count,
        })

    return {
        "pages_crawled": len(results),
        "frontier_remaining": len(frontier),
        "total_visited": len(visited),
        "results": results,
    }


# Crawl starting from a few seed URLs (limit to 5 pages for demo)
print("🕸️ Starting BFS crawl from seed URLs...\n")

output = crawl(
    seed_urls=["https://example.com", "https://httpbin.org"],
    max_pages=5,
)

print(f"\n📊 Crawl Summary:")
print(f"  Pages crawled: {output['pages_crawled']}")
print(f"  Frontier remaining: {output['frontier_remaining']}")
print(f"  URLs visited: {output['total_visited']}")

print(f"\n📝 Results:")
for r in output["results"]:
    print(f"  {r['url']}")
    print(f"    Text: {r['text_length']} chars | Discovered: {r['urls_discovered']} URLs | New: {r['new_urls_enqueued']}")
    print(f"    Preview: '{r['text_preview']}...'")

## 🤔 What's Wrong with This?

Our crawler works, but it has serious problems at scale:

| Problem | Impact |
|---------|--------|
| **Single process** | One machine, one thread. Can't crawl 10B pages in 5 days. |
| **No fault tolerance** | If the process crashes, all progress (visited set, frontier) is lost. |
| **No politeness** | Ignores robots.txt, no rate limiting per domain. Could get banned or overload servers. |
| **No dedup** | Different URLs can serve identical content — wastes time re-processing. |
| **In-memory state** | Visited set and frontier live in RAM. At 10B URLs, that's 100s of GB. |

**Next labs fix each of these:**
- **Lab 2:** Pipeline stages + retry (fault tolerance)
- **Lab 3:** robots.txt + domain rate limiting (politeness)
- **Lab 4:** URL dedup + content hashing (efficiency)